# 🌶️ Chilli Leaf Disease Detection — MobileNetV2 Transfer Learning
Run cells top to bottom. Runtime -> Change runtime type -> GPU (T4, free tier).

Dataset: *Chilli Leaf Disease Image Dataset for Classification and Early Diagnosis in Agriculture* (Mendeley, DOI 10.17632/tm3v4zmh7c.1).

## 1. Mount Google Drive & locate dataset zip

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# EDIT THIS to the path of the zip you uploaded to your Drive in README Step 2
DATASET_ZIP = '/content/drive/MyDrive/chilli_dataset.zip'


In [ ]:
import zipfile, os
EXTRACT_DIR = '/content/chilli_raw'
os.makedirs(EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile(DATASET_ZIP, 'r') as z:
    z.extractall(EXTRACT_DIR)

# Sanity check: print folders found and image counts per class
for root, dirs, files in os.walk(EXTRACT_DIR):
    imgs = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if imgs:
        print(root, '->', len(imgs), 'images')


## 2. Build a stratified train/val/test split (80/10/10)
The downloaded dataset is flat class folders, so we create the split ourselves. **Adjust `SOURCE_DIR` below** if step 1 printed a nested folder name (e.g. `.../Chilli_Leaf_Dataset/`).

In [ ]:
import glob, random, shutil

SOURCE_DIR = EXTRACT_DIR   # edit if the class folders are nested one level deeper
SPLIT_DIR = '/content/chilli_split'
random.seed(42)

class_dirs = [d for d in glob.glob(os.path.join(SOURCE_DIR, '*')) if os.path.isdir(d)]
# handle one extra nesting level automatically
if len(class_dirs) == 1:
    SOURCE_DIR = class_dirs[0]
    class_dirs = [d for d in glob.glob(os.path.join(SOURCE_DIR, '*')) if os.path.isdir(d)]

print('Classes found:', [os.path.basename(d) for d in class_dirs])

for split in ['train', 'val', 'test']:
    for d in class_dirs:
        os.makedirs(os.path.join(SPLIT_DIR, split, os.path.basename(d)), exist_ok=True)

for d in class_dirs:
    cls = os.path.basename(d)
    files = glob.glob(os.path.join(d, '*'))
    random.shuffle(files)
    n = len(files)
    n_train = int(n * 0.8)
    n_val = int(n * 0.1)
    splits = {
        'train': files[:n_train],
        'val': files[n_train:n_train + n_val],
        'test': files[n_train + n_val:]
    }
    for split, split_files in splits.items():
        for f in split_files:
            shutil.copy(f, os.path.join(SPLIT_DIR, split, cls, os.path.basename(f)))
    print(cls, '-> train:', len(splits['train']), 'val:', len(splits['val']), 'test:', len(splits['test']))


## 3. Data pipelines (resize, normalize, augment)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(SPLIT_DIR, 'train'), image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='int')
val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(SPLIT_DIR, 'val'), image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='int')
test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(SPLIT_DIR, 'test'), image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='int', shuffle=False)

class_names = train_ds.class_names
print('Classes (order matters, saved later):', class_names)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

def prep(ds, augment=False, shuffle=False):
    if shuffle:
        ds = ds.shuffle(1000)
    if augment:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds_p = prep(train_ds, augment=True, shuffle=True)
val_ds_p = prep(val_ds)
test_ds_p = prep(test_ds)


## 4. Class weights (handles class imbalance, see Phase 1 report)

In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

all_labels = np.concatenate([y.numpy() for x, y in train_ds])
class_weights_arr = compute_class_weight('balanced', classes=np.unique(all_labels), y=all_labels)
class_weight = {i: w for i, w in enumerate(class_weights_arr)}
print(class_weight)


## 5. Build MobileNetV2 transfer-learning model

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
base_model.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(class_names), activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()


## 6. Stage 1 — train the new head (base frozen)

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history1 = model.fit(train_ds_p, validation_data=val_ds_p, epochs=12,
                      class_weight=class_weight, callbacks=[early_stop])


## 7. Stage 2 — fine-tune the last layers of MobileNetV2

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history2 = model.fit(train_ds_p, validation_data=val_ds_p, epochs=10,
                      class_weight=class_weight, callbacks=[early_stop])


## 8. Evaluate: accuracy, precision, recall, F1, confusion matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

y_true = np.concatenate([y.numpy() for x, y in test_ds])
y_pred_probs = model.predict(test_ds_p)
y_pred = np.argmax(y_pred_probs, axis=1)

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion Matrix (test set)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png')
plt.show()

print('\nLook closely at which classes get confused with each other (e.g. Bacterial_Spot vs '
      'Cercospora_Leaf_Spot) — both are "spot" diseases and visually similar, so some confusion here is expected.')


## 9. Save the model + class names, then download
These two files are what you copy into `backend/model/` (README Step 4).

In [ ]:
import json
model.save('/content/chilli_disease_model.keras')
with open('/content/class_names.json', 'w') as f:
    json.dump(class_names, f)

from google.colab import files
files.download('/content/chilli_disease_model.keras')
files.download('/content/class_names.json')


## 10. (Recommended) Test on a few photos NOT from this dataset
Upload 3–5 real chilli leaf photos from elsewhere (your phone, Google Images of a clearly-labeled disease, etc.) here to sanity-check generalization before trusting the model in the app.

In [ ]:
from google.colab import files as up
from PIL import Image

uploaded = up.upload()
for fname in uploaded.keys():
    img = Image.open(fname).convert('RGB').resize(IMG_SIZE)
    arr = preprocess_input(np.array(img, dtype=np.float32))
    pred = model.predict(np.expand_dims(arr, 0), verbose=0)[0]
    idx = int(np.argmax(pred))
    print(fname, '->', class_names[idx], f'{pred[idx]*100:.1f}%')
